# Linux Command Basics · your first hour in the shell

Before you build edge systems, you need to be comfortable driving a Linux machine from the command line. This short pre-lab walks through the most useful commands from the course Linux Commands Guide, grouped by what you actually do with them: find your way around, work with files, read and search text, control processes, reach the network, and get help.

Work through it top to bottom. Run every code cell, read what comes back, and try the small exercises. Nothing here can harm the machine, so experiment freely.

The full one line reference for every command is in the [Linux Commands Guide](https://cs494.tail58eb9b.ts.net/static_files/linux-commands.pdf).

## How this notebook works · two places to run things

- **[Notebook cell]** means run the command right here with Shift+Enter. A cell starting with `!` runs one shell command; a cell starting with `%%bash` runs several lines.
- **[Terminal]** means open a terminal in JupyterLab (**File > New > Terminal**) and type the command there yourself. A few commands (like `top` and `htop`) take over the whole screen, so they only make sense in a real terminal.

Try both. The shell you reach from the notebook and the shell in the Terminal are the same machine.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

### Preflight · check your environment

In [ ]:
preflight([
    check("bash shell available", commandOnPath("bash"),
          hint="bash is the shell every command below runs in. It ships with the DGX Spark."),
    check("core commands present (grep)", commandOnPath("grep"),
          hint="grep should already be installed. Ask your instructor if this fails."),
    check("your home folder is writable", dirExists("~"),
          hint="You need a home folder to create the lab files. Ask your instructor if this fails."),
])

---
## Part 1 · Where am I, and who am I?

Every session starts with orientation. These commands answer "where am I standing and what machine is this?"

| Command | What it tells you |
|---|---|
| `whoami` | your username |
| `pwd` | the folder you are currently in (print working directory) |
| `ls` | what is in that folder (`ls -la` shows details and hidden files) |
| `uname -a` | which operating system and kernel this machine runs |
| `date` | the current date and time |

In [ ]:
!whoami

In [ ]:
!pwd

In [ ]:
!ls -la

In [ ]:
!uname -a

In [ ]:
!date "+%Y-%m-%d  %H:%M"

**[Notebook cell]** Make a folder to work in for the rest of the lab, then move into it. `mkdir -p` creates it (and does nothing if it already exists); `cd` changes directory.

In [ ]:
!mkdir -p ~/linuxBasicsLab
%cd ~/linuxBasicsLab
!pwd

---
## Part 2 · Working with files and folders

The daily verbs of the shell. You will create, copy, rename, and delete files here.

| Command | Does |
|---|---|
| `mkdir` | make a directory |
| `touch` | create an empty file (or update its timestamp) |
| `echo "text" > file` | write text into a file (`>>` appends instead of overwriting) |
| `cat` | print a file's contents |
| `cp` | copy a file |
| `mv` | move or rename a file |
| `rm` | remove a file (there is no undo, so look before you delete) |

**[Notebook cell]** Create some folders and files:

In [ ]:
%%bash
mkdir -p data notes
touch notes/todo.txt
echo "edge computing runs where the data is made" > notes/intro.txt
ls -R

**[Notebook cell]** Read a file back with `cat`:

In [ ]:
!cat notes/intro.txt

**[Notebook cell]** Copy it, rename the copy, then remove a file you no longer need:

In [ ]:
%%bash
cp notes/intro.txt data/intro-copy.txt   # copy
mv data/intro-copy.txt data/intro.txt    # rename
rm notes/todo.txt                        # delete
echo "--- data/ ---"; ls -l data
echo "--- notes/ ---"; ls -l notes

In [ ]:
checkpoint("Part 2 - files and folders", [
    check("working folder exists", dirExists("~/linuxBasicsLab"),
          hint="Run Part 1's mkdir/cd cell first."),
    check("notes/intro.txt has content", fileNonEmpty("~/linuxBasicsLab/notes/intro.txt"),
          hint="Run the echo cell that writes notes/intro.txt."),
    check("copy landed at data/intro.txt", fileExists("~/linuxBasicsLab/data/intro.txt"),
          hint="Run the cp/mv cell above."),
], successNote="You can create, read, copy, rename, and delete files. That is most of a working day in the shell.")

---
## Part 3 · Reading and searching text

Edge systems produce logs, and most of your debugging is reading and filtering text.

| Command | Does |
|---|---|
| `tail` | show the last lines of a file (`tail -n 5`); great for logs |
| `head` | the twin of `tail`: show the first lines |
| `grep` | find the lines that match a pattern |

**[Notebook cell]** First make a sample log with 50 sensor readings and one error line:

In [ ]:
%%bash
cd ~/linuxBasicsLab
for i in $(seq 1 50); do
  echo "$(date +%T) sensor reading $i value $((RANDOM % 100))"
done > data/sensor.log
echo "$(date +%T) ERROR pump offline" >> data/sensor.log
echo "wrote $(wc -l < data/sensor.log) lines to data/sensor.log"

**[Notebook cell]** Peek at the start and end of the log:

In [ ]:
!head -n 3 ~/linuxBasicsLab/data/sensor.log

In [ ]:
!tail -n 5 ~/linuxBasicsLab/data/sensor.log

**[Notebook cell]** Now find just the problem line with `grep`. This is the command you will reach for most often:

In [ ]:
!grep "ERROR" ~/linuxBasicsLab/data/sensor.log

**Try it:** change `ERROR` to `value 9` in the cell above and re-run it to see every reading whose value starts with 9.

In [ ]:
checkpoint("Part 3 - reading and searching text", [
    check("sensor.log was created", fileNonEmpty("~/linuxBasicsLab/data/sensor.log", minLines=50),
          hint="Run the cell that writes 50 readings into data/sensor.log."),
    check("the log contains an ERROR line", fileContains("~/linuxBasicsLab/data/sensor.log", "ERROR"),
          hint="Re-run the sample-log cell; it appends one ERROR line at the end."),
], successNote="tail, head, and grep let you find the one line that matters in a file of thousands.")

---
## Part 4 · Permissions · make a script and run it

Files carry permissions that decide who can read, write, or run them.

| Command | Does |
|---|---|
| `chmod` | change permissions (`chmod +x file` makes a file runnable) |
| `chown` | change which user owns a file (admins only; shown so you recognize it) |

**[Notebook cell]** Write a tiny shell script with `%%writefile`:

In [ ]:
%%writefile /home/jovyan/linuxBasicsLab/hello.sh
#!/bin/bash
echo "hello from $(whoami) on $(hostname)"

**[Notebook cell]** A fresh file is not executable yet. Make it runnable with `chmod +x`, then run it with `./`:

In [ ]:
%%bash
cd ~/linuxBasicsLab
chmod +x hello.sh
./hello.sh
ls -l hello.sh

Notice the `x` letters in `-rwxr-xr-x` from `ls -l`: those are the execute bits `chmod` just set. `chown` changes the *owner* of a file, but on a shared machine only an administrator can give your files to another user, so you will rarely run it yourself.

In [ ]:
checkpoint("Part 4 - permissions", [
    check("hello.sh exists", fileExists("~/linuxBasicsLab/hello.sh"),
          hint="Run the %%writefile cell that creates hello.sh."),
    check("hello.sh is executable and runs", commandSucceeds("~/linuxBasicsLab/hello.sh"),
          hint="Run the chmod +x cell so the script can execute."),
], successNote="You set a permission bit with chmod and ran your own script. That is how every tool on the box got runnable.")

---
## Part 5 · Processes and the system

A running program is a *process*. These commands show what is running and how the machine is doing.

| Command | Does |
|---|---|
| `ps` | list processes (`ps aux` shows every process) |
| `top` / `htop` | live, interactive view of processes (run these in a **Terminal**) |
| `kill` | stop a process by its PID (process id) |
| `df` | disk space free (`df -h` in human units) |
| `du` | how much space a folder uses (`du -sh folder/`) |

In [ ]:
!ps aux | head -n 5

In [ ]:
!df -h /

In [ ]:
!du -sh ~/linuxBasicsLab

**[Notebook cell]** See `kill` safely: start a harmless background job, find its PID, then stop it.

In [ ]:
%%bash
sleep 300 &                 # start a job in the background
pid=$!                      # $! is the PID of the last background job
echo "started background sleep with PID $pid"
ps -p $pid -o pid,comm      # confirm it is running
kill $pid                   # stop it
echo "sent kill to $pid"

**[Terminal]** Open **File > New > Terminal** and run `htop` (or `top`). You get a live dashboard of CPU, memory, and every process. Press `q` to quit. These are interactive, so they do not belong in a notebook cell.

---
## Part 6 · Reaching the network

Edge devices are defined by talking to other machines.

| Command | Does |
|---|---|
| `ping` | is another host reachable? |
| `wget` | download a file from the web |
| `ssh` | log in to another machine securely (you used this in Lab 01 to reach the DGX) |
| `scp` | copy files between machines over SSH |

In [ ]:
!ping -c 3 8.8.8.8 || echo "ping is blocked in this environment - that is OK"

**[Notebook cell]** Download a small file with `wget`:

In [ ]:
%%bash
cd ~/linuxBasicsLab
wget -q -O uic-robots.txt https://www.uic.edu/robots.txt \
  && { echo "downloaded:"; head uic-robots.txt; } \
  || echo "download blocked in this environment - that is OK"

`ssh` and `scp` need a second machine, so there is nothing to run here, but you already met them in **Lab 01**: `ssh user@host` opens a shell on a remote box, and `scp file user@host:/path/` copies a file to it. On the edge you use them constantly to reach devices in the field.

---
## Part 7 · Getting help without asking

The shell can explain itself. Learn these and you can figure out any command on your own.

| Command | Does |
|---|---|
| `man` | the full manual page for a command (press `q` to quit; here we pipe to `head`) |
| `whatis` | a one line description |
| `whereis` | where a command's program and manual live |
| `apt` / `sudo` | install software as an administrator (you cannot run these on the shared DGX; the instructor manages packages) |

In [ ]:
!whatis grep || true

In [ ]:
!whereis ls

In [ ]:
!man tar 2>/dev/null | head -n 20 || echo "man page not available here - try it in a Terminal"

`sudo` runs a command as the superuser, and `apt` installs system software (for example `sudo apt install nginx`). On the shared teaching machine you do **not** have `sudo`, and that is deliberate: the instructor manages what is installed so one student cannot break the box for everyone. You are seeing them so you recognize them in guides and Dockerfiles.

---
## Part 8 · Text power tools · awk and sed

Two classics that turn text into data. You will meet them again when you wrangle telemetry.

| Command | Does |
|---|---|
| `awk` | pull out columns and compute on them |
| `sed` | find and replace text in a stream |

**[Notebook cell]** `awk` prints chosen columns. Here `$1` is the time and `$NF` is the last field (the value):

In [ ]:
!awk '{print $1, $NF}' ~/linuxBasicsLab/data/sensor.log | head -n 5

**[Notebook cell]** `sed` substitutes text. This rewrites "sensor" to "device" on the way past (it does not change the file):

In [ ]:
!sed 's/sensor/device/' ~/linuxBasicsLab/data/sensor.log | head -n 3

---
## Part 9 · Bundle it up · tar

`tar` packs a whole folder into one file, the way you would zip a project to move or back it up.

| Command | Does |
|---|---|
| `tar -czf out.tar.gz folder/` | create a compressed archive of a folder |

In [ ]:
%%bash
cd ~/linuxBasicsLab
tar -czf ~/linuxBasicsLab.tar.gz .
ls -lh ~/linuxBasicsLab.tar.gz

In [ ]:
checkpoint("Part 9 - archiving with tar", [
    check("archive was created", fileExists("~/linuxBasicsLab.tar.gz"),
          hint="Run the tar cell above to pack the lab folder."),
], successNote="You packed a folder into one portable file. This is how projects and datasets move between machines.")

---
## Wrap up

You now have hands-on experience with the commands that carry every later lab:

- **Orient:** `whoami`, `pwd`, `ls`, `uname`, `date`
- **Files:** `mkdir`, `touch`, `echo >`, `cat`, `cp`, `mv`, `rm`
- **Text:** `head`, `tail`, `grep`, `awk`, `sed`
- **Permissions:** `chmod`
- **System:** `ps`, `top`/`htop`, `kill`, `df`, `du`
- **Network:** `ping`, `wget`, `ssh`, `scp`
- **Help:** `man`, `whatis`, `whereis`

Keep the [Linux Commands Guide](https://cs494.tail58eb9b.ts.net/static_files/linux-commands.pdf) open in a tab as a cheat sheet. When you are unsure what a command does, `man` it or `whatis` it before you run it.

**[Notebook cell]** Optional cleanup. Uncomment and run to remove the lab folder and archive when you are done.

In [ ]:
# import subprocess
# subprocess.run("rm -rf ~/linuxBasicsLab ~/linuxBasicsLab.tar.gz", shell=True)
# print("cleaned up")

### Lab scorecard

In [ ]:
labSummary("Linux Command Basics")

---

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Linux Command Basics")